# 🐢 Corner Time Loss Analysis
## `[RACE NAME]` · `[YEAR]` Formula 1 World Championship

**Where is a driver losing lap time?** This notebook aligns two laps by track distance,
computes the cumulative **delta time**, and breaks it down **corner by corner**.

- **REFERENCE lap** = the benchmark / target (usually the faster lap)
- **COMPARE lap** = the lap you're analysing
- Where the delta curve **rises**, the compare lap is **losing time**.

**Comparison ideas:**
- Same driver, their fastest lap vs an earlier/worse lap → *where did I improve?*
- Driver vs teammate (same car) → *pure driving difference*
- Driver vs pole sitter → *where's the gap to the front?*

---

In [ ]:
# ============================================================
# CONFIGURE
# ============================================================
YEAR       = 2026
GRAND_PRIX = 'Barcelona'
SESSION    = 'Q'        # 'Q' qualifying is ideal for clean single laps; 'R' also works

# REFERENCE lap (the benchmark — usually the faster one)
REF_DRIVER = 'VER'
REF_LAP    = 'fastest'  # 'fastest' or a lap number e.g. 18

# COMPARE lap (the lap being analysed)
COMP_DRIVER = 'LEC'
COMP_LAP    = 'fastest'  # 'fastest' or a lap number
# ============================================================

In [ ]:
import sys, os
from pathlib import Path

_here = Path(os.getcwd()).resolve()
_root = _here
for _ in range(5):
    if (_root / 'utils' / 'f1_helpers.py').exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from utils.f1_helpers import (
    setup, ensure_assets_dir,
    get_lap, analyze_corners, plot_delta_trace, plot_corner_breakdown,
)
import fastf1

setup('content/f1_cache')
ASSETS = ensure_assets_dir('assets')

session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION)
session.load(telemetry=True, weather=False)
print(f'\n✅ Loaded: {session.event["EventName"]} {YEAR} — {SESSION}')

In [ ]:
lap_ref  = get_lap(session, REF_DRIVER, REF_LAP)
lap_comp = get_lap(session, COMP_DRIVER, COMP_LAP)

print(f'REFERENCE  {REF_DRIVER}  lap {int(lap_ref["LapNumber"])}  →  {lap_ref["LapTime"]}')
print(f'COMPARE    {COMP_DRIVER}  lap {int(lap_comp["LapNumber"])}  →  {lap_comp["LapTime"]}')

---
## 1 · Run the delta analysis

In [ ]:
analysis = analyze_corners(session, lap_ref, lap_comp)

# Per-corner table — positive TimeLost = COMPARE driver lost time in that corner
analysis['df']

---
## 2 · Speed trace + delta curve

Red shading = compare lap losing time · Green = gaining. Dotted lines mark corners.

In [ ]:
fig1 = plot_delta_trace(session, analysis, lap_ref, lap_comp, ASSETS)
fig1.show()

---
## 3 · Corner-by-corner breakdown

In [ ]:
fig2 = plot_corner_breakdown(session, analysis, ASSETS)
fig2.show()

In [ ]:
# The 5 corners costing the compare driver the most time
worst = analysis['df'].sort_values('TimeLost', ascending=False).head(5)
print(f'Top 5 corners where {COMP_DRIVER} loses time vs {REF_DRIVER}:\n')
for _, row in worst.iterrows():
    print(f"  {row['Corner']:>5}  {row['TimeLost']:+.3f}s")

---
## 4 · LinkedIn Post Draft

In [ ]:
df = analysis['df']
total = df['TimeLost'].sum()
worst = df.sort_values('TimeLost', ascending=False).head(3)
best  = df.sort_values('TimeLost').head(2)
event_name = session.event['EventName']

worst_str = ', '.join(f"{r['Corner']} ({r['TimeLost']:+.2f}s)" for _, r in worst.iterrows())
best_str  = ', '.join(f"{r['Corner']} ({r['TimeLost']:+.2f}s)" for _, r in best.iterrows())

post = f"""
🔬 {event_name} {YEAR} — Where does {COMP_DRIVER} lose time to {REF_DRIVER}?

I aligned both fastest laps by track distance and tracked the delta corner by corner.

Total gap: {total:+.3f}s

🔴 Biggest losses for {COMP_DRIVER}:
{worst_str}

🟢 Where {COMP_DRIVER} actually gained:
{best_str}

💡 [Your read — e.g. is it traction on exit? braking confidence? top speed?]

Method: FastF1 telemetry + delta-time analysis in Python. Code on GitHub 👇

#Formula1 #F1 #DataAnalysis #DataScience #Motorsport #FastF1 #Python #{event_name.replace(' ', '')}
"""

print(post)

---
## Assets generated

| File | Use for |
|------|---------|
| `C1_delta_trace_*.png` | Speed + delta curve — the technical centrepiece |
| `C2_corner_breakdown_*.png` | Corner bar chart — most shareable / readable |

**Commit:**
```bash
git add 2026/R{round}_{location}/
git commit -m "feat: {event_name} corner time-loss analysis"
git push
```